<div style="padding:28px;border-radius:18px;background:linear-gradient(135deg,#0f172a,#1e3a5f);color:white">
  <div style="font-size:13px;letter-spacing:2px;font-weight:700">TOPIC 766 · TUTORIAL OF AGENTIC SYSTEMS</div>
  <h1 style="margin:8px 0 6px 0;font-size:34px">Notebook 0 — Building the M&A Learning Universe</h1>
  <div style="font-size:18px;opacity:.9">A synthetic 500-company dataset for a five-stage journey from agents to dynamic ecosystems</div>
</div>

### Introduction

Before we build an agent, we need to build the world in which that agent will learn to act. This notebook creates that world. The common setting for the entire tutorial is investment banking, specifically mergers and acquisitions. We will generate a synthetic universe of 500 companies distributed across continents and sectors. Each company will have a small but coherent financial profile, a strategic profile, and a compact set of unstructured documents such as financial-report excerpts, analyst notes, and M&A rumors. The objective is not realism for its own sake. The objective is pedagogy.

A useful learning environment must be rich enough to create genuine decisions but simple enough that a student can understand where every variable comes from. For that reason, the numerical data are generated from transparent rules with fixed random seeds. Revenues, margins, debt, cash, market capitalization, enterprise value, valuation multiples, and growth are related through understandable accounting and valuation identities. The qualitative layer is similarly controlled. GPT-5.2 is used only to generate reusable sector-specific language templates; the final company documents are assembled deterministically from those templates and the underlying company attributes.

The result is a deliberately small “M&A laboratory.” Later notebooks will use exactly the same universe. Notebook 1 will teach a single agent to use tools and skills to inspect it. Notebook 2 will introduce specialized agents and constellations. Notebook 3 will add feedback loops. Notebook 4 will allow the system to design its own constellation. Notebook 5 will allow constellations to interact. By keeping the world constant, we can see clearly what each new layer of agency contributes.

<div style="padding:20px;border-left:6px solid #2563eb;background:#eff6ff;border-radius:10px">
<b>What we need to understand and learn</b>
</div>

The most important lesson in this notebook is that a dataset for an agentic-system tutorial should not be treated as a passive collection of rows. It is an experimental environment. If the environment is badly designed, later agents may appear intelligent simply because the answers are obvious, or they may appear confused because the data are inconsistent for accidental reasons. We therefore need to engineer the information environment with the same care that we will later apply to the agents.

First, we need to distinguish <b>synthetic</b> from <b>arbitrary</b>. Synthetic data are invented, but they can still obey meaningful structure. A healthcare company can have different margin behavior from a utility; a rapidly growing technology company can carry a different valuation multiple from a mature industrial company. Enterprise value should approximately equal market capitalization plus debt minus cash. Net-debt-to-EBITDA should follow from those quantities rather than being sampled independently. These relationships create a world in which deterministic tools can calculate facts and agents can reason over them.

Second, we need two kinds of information. Structured data support precise calculations: filtering, ranking, valuation, leverage analysis, and peer comparison. Unstructured data support interpretation: strategic intent, management tone, rumors, analyst views, geographic ambitions, and possible red flags. M&A work naturally requires both. A company may look attractive numerically while a qualitative document introduces a complication. That disagreement becomes pedagogically valuable because it creates a reason for tools, multiple agents, evaluation, and later adaptation.

Third, we need controlled ambiguity. If every signal points in the same direction, there is little need for agency. Some companies therefore receive favorable financial profiles but cautionary textual signals; others have modest financial metrics but strong strategic complementarity. The data are not designed to have one obvious global answer. They are designed to create explainable trade-offs.

Fourth, we need reproducibility. Fixed seeds make the structured universe stable. The language model is not allowed to generate 500 independent companies because doing so would make the dataset costly, opaque, and difficult to reproduce. Instead, GPT-5.2 creates a small library of sector-specific sentence templates. Those templates are cached to Drive and reused. If the API is unavailable, the notebook has deterministic fallback templates, so the dataset can still be built.

Finally, we need a teacher’s view of the world. The notebook will create a separate pedagogical key containing latent M&A fit scores for candidate pairs. Later agents should not simply read this key. It exists so that, when we introduce evaluation and loops, we can compare an agent’s choices with a known synthetic benchmark. In other words, Notebook 0 does more than fabricate data: it creates the controlled experimental conditions for the entire five-stage journey.

A sixth lesson concerns **scale without complexity**. Five hundred companies sound substantial, but the learner should never need to hold five hundred complete corporate histories in mind. The dataset therefore uses many rows but few concepts. Every company follows the same schema, every document follows one of three recognizable evidence classes, and every later tool can operate through a small number of stable identifiers. This is a useful design pattern for teaching agentic systems: increase the size of the search space without simultaneously increasing the number of concepts the student must learn. The agent experiences a nontrivial universe, while the human can still inspect and understand the machinery that generates it.


### Code Unit 1 of 10 — Environment, Drive, secret, and model configuration

The first unit establishes the execution environment and the rules that every later notebook will inherit. Google Drive is mounted because the dataset must persist beyond a single Colab runtime. The destination is the common folder you specified: `/content/drive/MyDrive/Colab Notebooks/TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET`. The OpenAI credential is read from Colab Secrets under the exact name `OPENAI_API_KEY`; it is never printed and never written to disk. We also centralize the model name as `gpt-5.2`. Centralizing configuration is a small but important governance habit: if a future notebook changes model, path, or random seed, there is one explicit place to inspect. Finally, the cell creates a fixed random seed. This means that the structured part of the universe can be regenerated exactly, which is essential for teaching. If two students run the notebook, the same company IDs and financial relationships should be created rather than silently drifting from one run to another.

In [2]:
%pip -q install -U openai

from google.colab import drive, userdata
from openai import OpenAI
from pathlib import Path
import pandas as pd
import numpy as np
import json, re
from datetime import date

drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/Colab Notebooks/TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add OPENAI_API_KEY to the Colab Secrets panel before running this notebook.")

MODEL = "gpt-5.2"
SEED = 766
rng = np.random.default_rng(SEED)
client = OpenAI(api_key=OPENAI_API_KEY)

print("Dataset folder:", DATASET_DIR)
print("Model:", MODEL)
print("Random seed:", SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset folder: /content/drive/MyDrive/Colab Notebooks/TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET
Model: gpt-5.2
Random seed: 766


### Code Unit 2 of 10 — Define the ontology of the synthetic M&A world

An agent can reason only over concepts that exist in its environment, so this cell defines the ontology of our miniature investment-banking world. We specify six inhabited continents, representative countries, ten broad sectors, and several subsectors within each sector. The purpose is not to reproduce the full complexity of global capital markets. It is to provide enough diversity that later agents must deal with sector economics, geography, cross-border opportunities, and strategic adjacency. We also define sector-level priors for revenue scale, EBITDA margins, growth, leverage, and valuation. These priors are not forecasts; they are teaching parameters. They make a utility look structurally different from a software company and allow the student to see why valuation cannot be interpreted without context. Keeping these assumptions in one visible dictionary is crucial pedagogically. Nothing is hidden in a black box. Later, when an agent calls a valuation tool, the learner can trace the numbers back to the world-generation assumptions created here.

In [3]:
CONTINENTS = {
    "North America": ["United States", "Canada", "Mexico"],
    "South America": ["Brazil", "Chile", "Colombia", "Peru", "Argentina"],
    "Europe": ["United Kingdom", "Germany", "France", "Spain", "Italy", "Netherlands", "Sweden"],
    "Asia": ["Japan", "South Korea", "India", "Singapore", "Indonesia", "United Arab Emirates"],
    "Africa": ["South Africa", "Egypt", "Kenya", "Morocco", "Nigeria"],
    "Oceania": ["Australia", "New Zealand"],
}

SECTORS = {
    "Technology": ["Software", "Semiconductors", "IT Services", "Cybersecurity"],
    "Healthcare": ["Pharma", "Medical Devices", "Diagnostics", "Healthcare Services"],
    "Industrials": ["Automation", "Aerospace", "Logistics", "Engineering"],
    "Consumer": ["Retail", "Food & Beverage", "Apparel", "Consumer Services"],
    "Financials": ["Banking", "Insurance", "Payments", "Asset Management"],
    "Energy": ["Oil & Gas", "Renewables", "Energy Services", "Storage"],
    "Utilities": ["Electricity", "Water", "Gas Distribution", "Grid Services"],
    "Materials": ["Chemicals", "Metals", "Packaging", "Construction Materials"],
    "Real Estate": ["Industrial", "Office", "Residential", "Data Centers"],
    "Telecom & Media": ["Telecom", "Digital Media", "Broadcasting", "Connectivity"],
}

SECTOR_PRIORS = {
    "Technology":       dict(revenue=(500, 12000), margin=(0.14, 0.38), growth=(0.06, 0.30), leverage=(0.0, 2.5), multiple=(12, 26)),
    "Healthcare":       dict(revenue=(700, 15000), margin=(0.12, 0.32), growth=(0.03, 0.18), leverage=(0.5, 3.5), multiple=(10, 20)),
    "Industrials":      dict(revenue=(900, 20000), margin=(0.08, 0.22), growth=(-0.02, 0.12), leverage=(0.8, 4.0), multiple=(7, 14)),
    "Consumer":         dict(revenue=(800, 25000), margin=(0.06, 0.20), growth=(-0.03, 0.14), leverage=(0.5, 4.5), multiple=(7, 16)),
    "Financials":       dict(revenue=(1000, 30000), margin=(0.18, 0.40), growth=(0.00, 0.15), leverage=(0.0, 2.0), multiple=(8, 16)),
    "Energy":           dict(revenue=(1000, 30000), margin=(0.10, 0.30), growth=(-0.08, 0.18), leverage=(0.8, 4.5), multiple=(5, 12)),
    "Utilities":        dict(revenue=(900, 18000), margin=(0.18, 0.38), growth=(-0.01, 0.08), leverage=(2.0, 5.5), multiple=(7, 13)),
    "Materials":        dict(revenue=(800, 22000), margin=(0.08, 0.24), growth=(-0.06, 0.12), leverage=(0.8, 4.5), multiple=(5, 12)),
    "Real Estate":      dict(revenue=(300, 9000),  margin=(0.20, 0.55), growth=(-0.04, 0.14), leverage=(1.5, 6.0), multiple=(8, 18)),
    "Telecom & Media":  dict(revenue=(900, 25000), margin=(0.12, 0.35), growth=(-0.04, 0.12), leverage=(1.5, 5.5), multiple=(6, 14)),
}

print(f"{len(CONTINENTS)} continents | {sum(map(len, CONTINENTS.values()))} countries | {len(SECTORS)} sectors")

6 continents | 28 countries | 10 sectors


### Code Unit 3 of 10 — Generate the 500-company master table

This unit creates the identity layer of the universe. Every company receives a stable company ID, a synthetic name, country, continent, sector, subsector, founding year, and ownership status. The generator deliberately cycles across sectors before shuffling, which prevents the sample from accidentally concentrating in only a few industries. Geography is sampled independently enough to create cross-border possibilities, while the names are assembled from neutral prefixes and suffixes so that they resemble plausible corporations without intentionally reproducing real firms. The company table is the anchor for every other file: financials, strategic profiles, documents, and M&A candidate pairs all refer back to `company_id`. This teaches an important systems principle. Agents may interact with many tools and many representations, but the underlying environment needs stable identifiers and clean joins. Later notebooks will retrieve a company from one table, calculate valuation from another, and search text in a third. If identity is inconsistent, no amount of sophisticated reasoning can repair the foundation.

In [4]:
NAME_PREFIX = [
    "Aster", "Aurora", "Beacon", "BlueRiver", "Cedar", "Cobalt", "Crescent", "Delta",
    "Evergreen", "Frontier", "Granite", "Helix", "Horizon", "Juniper", "Keystone",
    "Lumen", "Meridian", "Northstar", "Orion", "Pioneer", "Redwood", "Summit",
    "Terra", "Vertex", "Westbridge"
]
NAME_SUFFIX = [
    "Systems", "Partners", "Industries", "Holdings", "Networks", "Technologies",
    "Group", "Solutions", "Capital", "Resources", "Global", "Enterprises"
]

def synthetic_name(i):
    a = NAME_PREFIX[(i * 7) % len(NAME_PREFIX)]
    b = NAME_SUFFIX[(i * 11 + 3) % len(NAME_SUFFIX)]
    return f"{a} {b} {i:03d}"

continent_names = list(CONTINENTS)
sector_names = list(SECTORS)
rows = []

for i in range(1, 501):
    sector = sector_names[(i - 1) % len(sector_names)]
    subsector = rng.choice(SECTORS[sector])
    continent = rng.choice(continent_names)
    country = rng.choice(CONTINENTS[continent])
    rows.append({
        "company_id": f"C{i:03d}",
        "company_name": synthetic_name(i),
        "country": country,
        "continent": continent,
        "sector": sector,
        "subsector": subsector,
        "founded_year": int(rng.integers(1950, 2018)),
        "ownership": rng.choice(["Public", "Private"], p=[0.72, 0.28]),
    })

companies = pd.DataFrame(rows).sort_values("company_id").reset_index(drop=True)

display(companies.head(10))
print("Companies:", len(companies))
display(companies["sector"].value_counts().sort_index().to_frame("companies"))

,company_id,company_name,country,continent,sector,subsector,founded_year,ownership
0,C001,Delta Industries 001,Brazil,South America,Technology,Cybersecurity,2000,Private
1,C002,Keystone Partners 002,New Zealand,Oceania,Healthcare,Healthcare Services,1967,Public
2,C003,Summit Systems 003,New Zealand,Oceania,Industrials,Aerospace,2003,Public
3,C004,BlueRiver Enterprises 004,India,Asia,Consumer,Retail,1951,Public
4,C005,Granite Global 005,Indonesia,Asia,Financials,Banking,1967,Private
5,C006,Northstar Resources 006,Chile,South America,Energy,Energy Services,1961,Public
6,C007,Westbridge Capital 007,Australia,Oceania,Utilities,Water,2011,Public
7,C008,Crescent Solutions 008,United Arab Emirates,Asia,Materials,Packaging,1989,Public
8,C009,Juniper Group 009,Mexico,North America,Real Estate,Industrial,2000,Public
9,C010,Redwood Technologies 010,Sweden,Europe,Telecom & Media,Telecom,1989,Public


Companies: 500


,companies
sector,
Consumer,50
Energy,50
Financials,50
Healthcare,50
Industrials,50
Materials,50
Real Estate,50
Technology,50
Telecom & Media,50


### Code Unit 4 of 10 — Create coherent financial statements and valuation variables

Here we give each synthetic company a compact financial profile. The central teaching principle is coherence. Rather than drawing revenue, EBITDA, debt, cash, enterprise value, and valuation ratios independently, the code creates them in a sequence that preserves relationships. Revenue is sampled from a sector-specific scale. EBITDA follows from revenue multiplied by a sector-appropriate margin. Debt is linked to EBITDA through leverage, while cash is drawn as a fraction of revenue. Enterprise value is generated from EBITDA and an EV/EBITDA multiple; market capitalization is then derived from enterprise value minus debt plus cash. If that calculation becomes implausibly small, the code floors it at a modest positive value. Finally, the notebook recalculates leverage and valuation ratios from the generated quantities. This structure allows later deterministic tools to verify the same mathematics an analyst would use. The dataset therefore becomes useful not merely for retrieval, but for teaching the difference between observed inputs, calculated metrics, and model-based interpretation.

In [5]:
financial_rows = []

for _, c in companies.iterrows():
    p = SECTOR_PRIORS[c["sector"]]
    revenue = rng.uniform(*p["revenue"])
    margin = rng.uniform(*p["margin"])
    ebitda = revenue * margin
    growth = rng.uniform(*p["growth"])
    leverage = rng.uniform(*p["leverage"])
    debt = max(0.0, ebitda * leverage)
    cash = revenue * rng.uniform(0.02, 0.18)
    ev_multiple = rng.uniform(*p["multiple"])
    enterprise_value = ebitda * ev_multiple
    market_cap = max(enterprise_value - debt + cash, 0.20 * enterprise_value)

    financial_rows.append({
        "company_id": c["company_id"],
        "revenue_usd_m": round(revenue, 1),
        "ebitda_usd_m": round(ebitda, 1),
        "ebitda_margin_pct": round(100 * ebitda / revenue, 2),
        "revenue_growth_pct": round(100 * growth, 2),
        "debt_usd_m": round(debt, 1),
        "cash_usd_m": round(cash, 1),
        "market_cap_usd_m": round(market_cap, 1),
        "enterprise_value_usd_m": round(enterprise_value, 1),
        "ev_ebitda": round(enterprise_value / ebitda, 2),
        "net_debt_ebitda": round((debt - cash) / ebitda, 2),
    })

financials = pd.DataFrame(financial_rows)
display(financials.head())
print("Median EV/EBITDA:", financials["ev_ebitda"].median())
print("Median growth:", financials["revenue_growth_pct"].median(), "%")

,company_id,revenue_usd_m,ebitda_usd_m,ebitda_margin_pct,revenue_growth_pct,debt_usd_m,cash_usd_m,market_cap_usd_m,enterprise_value_usd_m,ev_ebitda,net_debt_ebitda
0,C001,1968.5,731.6,37.17,16.35,1500.5,104.6,10258.6,11654.6,15.93,1.91
1,C002,11151.7,2647.6,23.74,17.79,7305.0,1550.9,46801.7,52555.9,19.85,2.17
2,C003,5538.3,1149.7,20.76,5.93,1464.3,575.7,9646.5,10535.1,9.16,0.77
3,C004,4001.9,428.1,10.70,-2.72,1763.0,575.9,3021.0,4208.1,9.83,2.77
4,C005,12965.2,4919.8,37.95,10.09,8389.4,1651.8,41603.8,48341.5,9.83,1.37


Median EV/EBITDA: 11.34
Median growth: 6.45 %


### Code Unit 5 of 10 — Add strategic profiles that cannot be reduced to financial ratios

M&A decisions are not valuation screens with a few extra adjectives. They depend on strategic fit, geographic ambitions, operating capabilities, ownership constraints, regulatory exposure, and management appetite. This cell therefore creates a second structured layer containing qualitative but machine-readable attributes. Each company receives a business-model description, a strategic strength, a strategic weakness, a geographic posture, acquisition appetite, cross-border openness, regulatory sensitivity, and integration complexity. These variables will later support several different tools and roles. A financial agent may care about leverage and valuation; a strategy agent may care about product adjacency and geography; a risk agent may focus on regulatory sensitivity and integration complexity. The reason to separate strategic profiles from financials is pedagogical: students should see that different analytical questions require different representations of the same company. In later notebooks, this separation will make specialization meaningful because different agents can consult different evidence while still referring to the same stable company identity.

In [6]:
STRENGTHS = [
    "strong distribution network", "high recurring revenue", "proprietary technology",
    "trusted brand", "low-cost operating model", "scarce licenses", "dense customer network",
    "strong regional market share", "high switching costs", "specialized talent base"
]
WEAKNESSES = [
    "customer concentration", "elevated leverage", "slow international expansion",
    "legacy systems", "margin pressure", "regulatory dependence", "cyclical demand",
    "fragmented operations", "limited scale", "capital intensity"
]
BUSINESS_MODELS = [
    "subscription-led", "transaction-led", "asset-heavy", "project-based",
    "platform-based", "distribution-led", "regulated infrastructure",
    "consumer brand", "enterprise solutions", "hybrid services"
]

strategic_rows = []
for _, c in companies.iterrows():
    strategic_rows.append({
        "company_id": c["company_id"],
        "business_model": rng.choice(BUSINESS_MODELS),
        "strategic_strength": rng.choice(STRENGTHS),
        "strategic_weakness": rng.choice(WEAKNESSES),
        "geographic_posture": rng.choice(["Domestic", "Regional", "Multi-region", "Global"], p=[0.22, 0.32, 0.28, 0.18]),
        "acquisition_appetite": rng.choice(["Low", "Moderate", "High"], p=[0.28, 0.46, 0.26]),
        "cross_border_openness": rng.choice(["Low", "Moderate", "High"], p=[0.22, 0.47, 0.31]),
        "regulatory_sensitivity": rng.choice(["Low", "Moderate", "High"], p=[0.35, 0.43, 0.22]),
        "integration_complexity": rng.choice(["Low", "Moderate", "High"], p=[0.25, 0.50, 0.25]),
    })

strategic_profiles = pd.DataFrame(strategic_rows)
display(strategic_profiles.head())

,company_id,business_model,strategic_strength,strategic_weakness,geographic_posture,acquisition_appetite,cross_border_openness,regulatory_sensitivity,integration_complexity
0,C001,distribution-led,specialized talent base,elevated leverage,Multi-region,Moderate,Moderate,Moderate,Low
1,C002,regulated infrastructure,low-cost operating model,capital intensity,Global,Moderate,Low,High,Moderate
2,C003,platform-based,dense customer network,fragmented operations,Global,Moderate,Low,Moderate,Moderate
3,C004,enterprise solutions,trusted brand,cyclical demand,Domestic,Low,Moderate,High,Moderate
4,C005,project-based,proprietary technology,margin pressure,Domestic,Moderate,Moderate,High,Moderate


### Code Unit 6 of 10 — Use GPT-5.2 to create a small reusable language-template library

This is the only point where the language model participates in building the information environment. Importantly, GPT-5.2 is not asked to invent 500 complete companies. That would make the dataset expensive, difficult to reproduce, and pedagogically opaque. Instead, the model receives the ten sectors and is asked to produce a compact JSON library containing short sentence templates for three document types: financial-report excerpts, analyst notes, and M&A rumors. Each template contains placeholders such as company name, geography, strategic strength, or weakness. The output is cached in the common Drive folder. On subsequent runs, the notebook reuses the cached template file rather than calling the API again. If the call fails or the returned JSON cannot be parsed, deterministic fallback templates are used. This design illustrates a principle we will repeatedly use in agentic systems: reserve the language model for tasks that benefit from language flexibility, while keeping deterministic structure, identity, calculations, and validation outside the model.

In [7]:
TEMPLATE_FILE = DATASET_DIR / "narrative_templates.json"

FALLBACK = {
    "management": [
        "{company} reports continued focus on {strength} while expanding selectively in {continent}.",
        "Management highlights {strength} as a priority and acknowledges {weakness} as an execution challenge.",
        "{company} expects disciplined growth in {subsector}, with capital allocation focused on operational resilience."
    ],
    "analyst": [
        "Analysts view {strength} as strategically valuable, although {weakness} may complicate near-term execution.",
        "{company} could attract strategic interest from buyers seeking exposure to {subsector} in {continent}.",
        "The current profile combines credible strategic assets with identifiable integration risks."
    ],
    "rumor": [
        "Market sources say {company} has appeared in informal strategic discussions; no transaction has been confirmed.",
        "Unverified industry chatter links {company} with potential consolidation in {subsector}.",
        "Banking sources describe preliminary interest around {company}, although timing and counterparties remain uncertain."
    ]
}

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))

if TEMPLATE_FILE.exists():
    template_library = json.loads(TEMPLATE_FILE.read_text())
    print("Loaded cached GPT template library.")
else:
    prompt = f"""
Create a compact pedagogical template library for a synthetic M&A dataset.
Sectors: {list(SECTORS.keys())}

Return ONLY valid JSON. For each sector, provide exactly:
- 3 financial_report_excerpt sentence templates
- 3 analyst_note sentence templates
- 3 rumor sentence templates

Each sentence must be 18-35 words and may use only these placeholders:
{{company}}, {{country}}, {{continent}}, {{sector}}, {{subsector}}, {{strength}}, {{weakness}}.

Rumors must be clearly uncertain and must never assert a completed transaction.
Keep the language realistic but generic. This is synthetic educational data.
"""
    try:
        response = client.responses.create(model=MODEL, input=prompt)
        template_library = extract_json(response.output_text)
        TEMPLATE_FILE.write_text(json.dumps(template_library, indent=2))
        print("Created and cached GPT-5.2 template library.")
    except Exception as exc:
        print("GPT template generation failed; using deterministic fallback.")
        print("Reason:", type(exc).__name__)
        template_library = {
            sector: {
                "financial_report_excerpt": FALLBACK["management"],
                "analyst_note": FALLBACK["analyst"],
                "rumor": FALLBACK["rumor"],
            }
            for sector in SECTORS
        }
        TEMPLATE_FILE.write_text(json.dumps(template_library, indent=2))

print("Template sectors:", len(template_library))

Created and cached GPT-5.2 template library.
Template sectors: 10


### Code Unit 7 of 10 — Generate the unstructured document corpus

Now we convert the template library into the textual evidence that future agents will search. Each company receives three short documents: one financial-report excerpt, one analyst note, and one rumor. The templates are populated with company-specific attributes, so the text remains connected to the structured world rather than floating independently from it. We also attach simple metadata: document ID, document type, a synthetic observation period, and a source-reliability category. Reliability is deliberately different across document types. Financial-report excerpts is authoritative about what management says but may be promotional; analyst notes are interpretive; rumors are intentionally uncertain. Later notebooks can choose whether to expose reliability directly to the agent or make the agent infer it from document type. This corpus gives us the central tension of the tutorial: a deterministic tool can calculate EV/EBITDA exactly, but no formula alone can decide how much weight to give an unconfirmed rumor or a strategic statement.

In [8]:
def sector_templates(sector):
    entry = template_library.get(sector, {})
    return {
        "financial_report_excerpt": entry.get("financial_report_excerpt", FALLBACK["management"]),
        "analyst_note": entry.get("analyst_note", FALLBACK["analyst"]),
        "rumor": entry.get("rumor", FALLBACK["rumor"]),
    }

merged = companies.merge(strategic_profiles, on="company_id")
document_rows = []
doc_counter = 1

for _, r in merged.iterrows():
    mapping = {
        "company": r["company_name"],
        "country": r["country"],
        "continent": r["continent"],
        "sector": r["sector"],
        "subsector": r["subsector"],
        "strength": r["strategic_strength"],
        "weakness": r["strategic_weakness"],
    }
    templates = sector_templates(r["sector"])
    specs = [
        ("financial_report_excerpt", "High"),
        ("analyst_note", "Moderate"),
        ("rumor", "Low"),
    ]
    for dtype, reliability in specs:
        template = rng.choice(templates[dtype])
        text = template.format(**mapping)
        document_rows.append({
            "document_id": f"D{doc_counter:04d}",
            "company_id": r["company_id"],
            "document_type": dtype,
            "observation_period": rng.choice(["2026-H1", "2026-H2", "2027-H1"]),
            "source_reliability": reliability,
            "text": text,
        })
        doc_counter += 1

documents = pd.DataFrame(document_rows)
display(documents.head(9))
print("Documents:", len(documents))
print(documents["document_type"].value_counts())

,document_id,company_id,document_type,observation_period,source_reliability,text
0,D0001,C001,financial_report_excerpt,2026-H1,High,Delta Industries 001 reported steady momentum ...
1,D0002,C001,analyst_note,2026-H1,Moderate,Our channel checks suggest Delta Industries 00...
2,D0003,C001,rumor,2026-H2,Low,There is speculative talk that Delta Industrie...
3,D0004,C002,financial_report_excerpt,2026-H2,High,Keystone Partners 002 said results were suppor...
4,D0005,C002,analyst_note,2027-H1,Moderate,Our view is that Keystone Partners 002 has cre...
5,D0006,C002,rumor,2027-H1,Low,Unconfirmed commentary suggests Keystone Partn...
6,D0007,C003,financial_report_excerpt,2027-H1,High,Summit Systems 003 highlighted progress initia...
7,D0008,C003,analyst_note,2026-H2,Moderate,Our analysis suggests Summit Systems 003 has i...
8,D0009,C003,rumor,2027-H1,Low,Some market participants speculate Summit Syst...


Documents: 1500
document_type
financial_report_excerpt    500
analyst_note                500
rumor                       500
Name: count, dtype: int64


### Code Unit 8 of 10 — Construct a teacher-only M&A candidate universe and latent fit scores

A tutorial becomes much more powerful when later agent decisions can be evaluated against a known experimental benchmark. This unit creates candidate buyer-target pairs and computes a synthetic “teacher fit score.” The score is not meant to represent the true economics of M&A. It is an explicit pedagogical rule combining sector adjacency, geographic complementarity, growth, valuation, leverage, strategic openness, and regulatory friction. Because the rule is visible here, we know what the synthetic world rewards. Later student-facing agents should not simply read the resulting teacher key; they should reach recommendations through their own tools and reasoning. The key is reserved for evaluation, especially in Notebook 3 when feedback loops are introduced. We also avoid generating all 249,500 possible ordered pairs. Instead, each company receives a small sample of plausible counterparties, which keeps the file compact and the problem understandable. This gives us controlled “ground truth” without pretending that real M&A has a single correct answer.

In [9]:
full = companies.merge(financials, on="company_id").merge(strategic_profiles, on="company_id")

pair_rows = []
for _, buyer in full.iterrows():
    pool = full[full["company_id"] != buyer["company_id"]].sample(
        n=8, random_state=SEED + int(buyer["company_id"][1:])
    )
    for _, target in pool.iterrows():
        same_sector = 1.0 if buyer["sector"] == target["sector"] else 0.0
        same_continent = 1.0 if buyer["continent"] == target["continent"] else 0.0
        cross_border_bonus = 1.0 if buyer["country"] != target["country"] else 0.0
        growth_score = np.clip((target["revenue_growth_pct"] + 10) / 40, 0, 1)
        valuation_score = np.clip((22 - target["ev_ebitda"]) / 17, 0, 1)
        leverage_score = np.clip((5 - max(target["net_debt_ebitda"], 0)) / 5, 0, 1)
        openness = {"Low": 0.2, "Moderate": 0.6, "High": 1.0}[buyer["cross_border_openness"]]
        reg_penalty = {"Low": 0.0, "Moderate": 0.15, "High": 0.30}[target["regulatory_sensitivity"]]

        fit = (
            0.25 * same_sector
            + 0.10 * same_continent
            + 0.10 * cross_border_bonus * openness
            + 0.18 * growth_score
            + 0.17 * valuation_score
            + 0.15 * leverage_score
            + 0.05 * (1 - reg_penalty)
        )

        pair_rows.append({
            "buyer_id": buyer["company_id"],
            "target_id": target["company_id"],
            "teacher_fit_score": round(100 * float(np.clip(fit, 0, 1)), 2),
            "same_sector": bool(same_sector),
            "cross_border": buyer["country"] != target["country"],
        })

teacher_key = pd.DataFrame(pair_rows).sort_values(
    ["buyer_id", "teacher_fit_score"], ascending=[True, False]
).reset_index(drop=True)

display(teacher_key.head(12))
print("Candidate pairs:", len(teacher_key))

,buyer_id,target_id,teacher_fit_score,same_sector,cross_border
0,C001,C311,68.11,True,False
1,C001,C061,65.05,True,True
2,C001,C298,49.51,False,True
3,C001,C153,49.04,False,True
4,C001,C165,40.01,False,True
5,C001,C412,39.13,False,True
6,C001,C299,33.88,False,True
7,C001,C440,31.31,False,True
8,C002,C422,68.53,True,True
9,C002,C095,47.63,False,False


Candidate pairs: 4000


### Code Unit 9 of 10 — Save the complete learning environment and its documentation

This unit turns the in-memory tables into a reusable course asset. The four student-facing files—companies, financials, strategic profiles, and documents—are written to the common Drive folder. The teacher-only M&A pair key is stored separately and labeled accordingly. We also create a machine-readable data dictionary, a manifest that records version, seed, model, row counts, and generation date, and a short README explaining the intended pedagogical use. This documentation matters because agentic systems quickly accumulate components. Without provenance, future notebooks could silently depend on a file whose meaning has changed. By writing the schema and generation assumptions alongside the data, we make the environment inspectable. The manifest also creates a simple contract between Notebook 0 and Notebook 1: before an agent is allowed to reason over the universe, it should be able to verify that the expected files and row counts exist. In later stages this idea will evolve into stronger execution contracts and governance.

In [10]:
files = {
    "companies.csv": companies,
    "financials.csv": financials,
    "strategic_profiles.csv": strategic_profiles,
    "documents.csv": documents,
    "teacher_mna_key.csv": teacher_key,
}

for filename, df in files.items():
    df.to_csv(DATASET_DIR / filename, index=False)

data_dictionary = {
    "companies.csv": {c: str(t) for c, t in companies.dtypes.items()},
    "financials.csv": {c: str(t) for c, t in financials.dtypes.items()},
    "strategic_profiles.csv": {c: str(t) for c, t in strategic_profiles.dtypes.items()},
    "documents.csv": {c: str(t) for c, t in documents.dtypes.items()},
    "teacher_mna_key.csv": {c: str(t) for c, t in teacher_key.dtypes.items()},
}
(DATASET_DIR / "data_dictionary.json").write_text(json.dumps(data_dictionary, indent=2))

manifest = {
    "tutorial": "TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS",
    "notebook": "NB00_BUILD_MA_DATASET",
    "dataset_version": "1.0",
    "seed": SEED,
    "language_model": MODEL,
    "companies": len(companies),
    "documents": len(documents),
    "candidate_pairs": len(teacher_key),
    "generated_on": str(date.today()),
    "synthetic_data_only": True,
}
(DATASET_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

readme_lines = [
    "# TOPIC 766 — Synthetic M&A Learning Universe",
    "",
    "This folder is generated by **NB00_BUILD_MA_DATASET.ipynb**.",
    "",
    "## Purpose",
    "A minimalistic, synthetic and reproducible information environment for teaching",
    "agentic systems through an investment-banking / M&A case.",
    "",
    "## Student-facing files",
    "- companies.csv",
    "- financials.csv",
    "- strategic_profiles.csv",
    "- documents.csv",
    "",
    "## Teacher/evaluation file",
    "- teacher_mna_key.csv",
    "",
    "The teacher key is intended for later evaluation exercises and should not be used",
    "as an input shortcut by student-facing agents.",
    "",
    "## Reproducibility",
    f"- Seed: {SEED}",
    f"- Language model for narrative template generation: {MODEL}",
    f"- Number of companies: {len(companies)}",
    f"- Number of documents: {len(documents)}",
    "",
    "All company names and data are synthetic.",
]
(DATASET_DIR / "README.md").write_text("\n".join(readme_lines))

print("Saved dataset assets:")
for p in sorted(DATASET_DIR.iterdir()):
    print(" -", p.name)

Saved dataset assets:
 - README.md
 - companies.csv
 - data_dictionary.json
 - documents.csv
 - financials.csv
 - manifest.json
 - narrative_templates.json
 - strategic_profiles.csv
 - teacher_mna_key.csv


### Code Unit 10 of 10 — Validate the dataset and create the bridge to Notebook 1

The final unit is not another generation step. It is a gate. Before the tutorial moves from environment construction to agency, we verify that the world satisfies its own contract. The checks confirm that there are exactly 500 unique companies, exactly three unstructured documents per company, complete company IDs across all core tables, and no missing values in the variables that later tools will depend on. We also recompute enterprise value from market capitalization, debt, and cash and report the typical reconciliation error. Because market capitalization is floored in rare cases to avoid nonsensical negative equity values, the identity is treated as an approximate diagnostic rather than a rigid equality. Finally, the cell prints a compact example company with its financial profile and associated documents. That preview is pedagogically important: the learner should be able to look at one company and understand the information world before any agent touches it. Once these checks pass, Notebook 0 has done its job and Notebook 1 can introduce the primitives of action.

In [11]:
assert len(companies) == 500, "Expected exactly 500 companies."
assert companies["company_id"].is_unique, "Company IDs must be unique."
assert len(financials) == 500 and len(strategic_profiles) == 500
assert set(financials["company_id"]) == set(companies["company_id"])
assert set(strategic_profiles["company_id"]) == set(companies["company_id"])
assert len(documents) == 1500, "Expected exactly three documents per company."
assert documents.groupby("company_id").size().eq(3).all()
assert financials.isna().sum().sum() == 0
assert strategic_profiles.isna().sum().sum() == 0

reconstructed_ev = (
    financials["market_cap_usd_m"] + financials["debt_usd_m"] - financials["cash_usd_m"]
)
median_abs_ev_gap = (financials["enterprise_value_usd_m"] - reconstructed_ev).abs().median()

sample_id = "C001"
sample = (
    companies[companies["company_id"] == sample_id]
    .merge(financials, on="company_id")
    .merge(strategic_profiles, on="company_id")
)
sample_docs = documents[documents["company_id"] == sample_id][
    ["document_type", "source_reliability", "text"]
]

print("✓ DATASET VALIDATION PASSED")
print("✓ Companies:", len(companies))
print("✓ Documents:", len(documents))
print("✓ Teacher candidate pairs:", len(teacher_key))
print("✓ Median absolute EV reconciliation gap (USD m):", round(float(median_abs_ev_gap), 2))
print("\nExample company:")
display(sample.T)
print("\nAssociated unstructured evidence:")
display(sample_docs)

print("\nNEXT STAGE:")
print("Notebook 1 — PRIMITIVES: Agents → Tools → Skills")
print("Question: How does an individual M&A agent acquire the ability to act?")

✓ DATASET VALIDATION PASSED
✓ Companies: 500
✓ Documents: 1500
✓ Teacher candidate pairs: 4000
✓ Median absolute EV reconciliation gap (USD m): 0.0

Example company:


,0
company_id,C001
company_name,Delta Industries 001
country,Brazil
continent,South America
sector,Technology
subsector,Cybersecurity
founded_year,2000
ownership,Private
revenue_usd_m,1968.5
ebitda_usd_m,731.6



Associated unstructured evidence:


,document_type,source_reliability,text
0,financial_report_excerpt,High,Delta Industries 001 reported steady momentum ...
1,analyst_note,Moderate,Our channel checks suggest Delta Industries 00...
2,rumor,Low,There is speculative talk that Delta Industrie...



NEXT STAGE:
Notebook 1 — PRIMITIVES: Agents → Tools → Skills
Question: How does an individual M&A agent acquire the ability to act?


<div style="padding:20px;border-left:6px solid #16a34a;background:#f0fdf4;border-radius:10px">
<b>Conclusion — From a world of data to an agent capable of acting</b>
</div>

Notebook 0 has created the common experimental world for the entire tutorial. That distinction matters. We have not yet built an agentic system. We have built the environment against which increasingly sophisticated forms of agency can be observed and compared. The universe contains 500 synthetic companies with stable identities, coherent financial variables, structured strategic attributes, and three categories of short unstructured evidence. It also contains a teacher-only M&A candidate key that gives us a controlled benchmark for later evaluation.

Several design choices will remain important throughout the course. The first is the separation between deterministic and probabilistic work. Company identities, financial relationships, row counts, joins, and validation rules are deterministic. GPT-5.2 contributes only to the language layer, where variation in wording is useful. This prevents the language model from becoming an invisible generator of facts. The second is reproducibility. A fixed seed means that the same structured universe can be rebuilt, inspected, and challenged. The third is intentional ambiguity. Financial attractiveness, strategic fit, and textual signals do not always align. That is not a defect; it is what creates the need for reasoning and, eventually, for multiple specialized agents and feedback.

The dataset also introduces a governance principle that will become increasingly important as the architecture grows: every stage should have an explicit contract. Notebook 0 ends with a validation gate. We know what files exist, how many records they contain, what schema they use, and whether the core relationships are internally consistent. Later notebooks will apply the same discipline to tools, agent outputs, inter-agent messages, loops, and constellation formation.

Most importantly, the learner can now see the information environment directly. Before any model is asked to recommend a target, a student can inspect a company, calculate its valuation, read its financial-report excerpts, compare an analyst note with a rumor, and understand where each piece of evidence came from. That transparency is essential to the pedagogical objective of the tutorial. We are not trying to impress the learner with autonomous behavior whose internal structure is hidden. We are building agency progressively from components that remain visible.

The next notebook changes the question completely. We stop asking, “What information exists?” and begin asking, “How can a system use that information to act?” Notebook 1 will introduce the three primitives: **agents, tools, and skills**. The M&A setting will remain constant. A single investment-banking agent will receive a task such as identifying or comparing acquisition candidates. It will not be allowed to magically know the answer. Instead, it will have to call deterministic tools, interpret their observations, and combine them through reusable skills.

That transition is the first genuine step in the ladder:

**Environment → Agent → Tools → Skills.**

The central question for Notebook 1 will therefore be:

> **How does an individual agent acquire the ability to act inside a known world?**

Notebook 0 also gives us a baseline against which architectural progress can be judged. If a later system produces a better recommendation, we should be able to ask whether the improvement came from better tools, specialization, feedback, dynamic organization, or information sharing. Because the underlying company universe remains fixed, the pedagogical experiment has a stable reference point. The increasing sophistication will come from the **architecture of agency**, not from quietly changing the problem underneath the learner.
